<a href="https://colab.research.google.com/github/stivenalvarez/Proyecto-PLN/blob/main/Procesamiento_de_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Carga de un dataset representativo de Comentarios de YouTube (PLN)
data = {
    "id_comentario": [101, 102, 103, 104, 105],
    "usuario": ["tech_user99", "dev_carlos", "ana_ai", "code_master", "laura_m"],
    "comentario": [
        "¡Excelente explicación del algoritmo de NLP! Me ayudó mucho para mi examen. http://youtube.com/watch?v=123",
        "El video es muy largo y la explicación de los transformers fue demasiado rápida y confusa.",
        "¿Alguien sabe qué librería de Python es mejor para sentiment analysis en 2026? Saludos desde Colombia.",
        "No me funcionó el código de Hugging Face, me sale un error de versión en Colab. Pésimo tutorial.",
        "Increíble contenido sobre Inteligencia Artificial y PLN. Muy bien estructurado y fácil de seguir."
    ],
    "categoria": ["Educación", "Crítica", "Pregunta", "Error", "Educación"]
}

df = pd.DataFrame(data)
df.rename(columns={"comentario": "texto_original"}, inplace=True)
print(df[["id_comentario", "usuario", "texto_original"]])

   id_comentario      usuario  \
0            101  tech_user99   
1            102   dev_carlos   
2            103       ana_ai   
3            104  code_master   
4            105      laura_m   

                                      texto_original  
0  ¡Excelente explicación del algoritmo de NLP! M...  
1  El video es muy largo y la explicación de los ...  
2  ¿Alguien sabe qué librería de Python es mejor ...  
3  No me funcionó el código de Hugging Face, me s...  
4  Increíble contenido sobre Inteligencia Artific...  


In [6]:
import re
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Instancias para procesamiento
stop_words_es = set(stopwords.words('spanish'))
stemmer = SnowballStemmer("spanish")

def normalizar_texto(texto):
    # 1. Convertir a minúsculas
    texto = texto.lower()
    # 2. Eliminar URLs de YouTube y web
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    # 3. Eliminar caracteres especiales y puntuación (conservando letras, números y espacios)
    texto = re.sub(r'[^\w\s]', '', texto)
    # 4. Eliminar espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def eliminar_stopwords(texto):
    palabras = texto.split()
    palabras_filtradas = [word for word in palabras if word not in stop_words_es]
    return " ".join(palabras_filtradas)

def aplicar_stemming(texto):
    palabras = texto.split()
    palabras_stem = [stemmer.stem(word) for word in palabras]
    return " ".join(palabras_stem)

# Aplicación secuencial de las transformaciones
df['texto_normalizado'] = df['texto_original'].apply(normalizar_texto)
df['texto_sin_stopwords'] = df['texto_normalizado'].apply(eliminar_stopwords)
df['texto_stemmed'] = df['texto_sin_stopwords'].apply(aplicar_stemming)

# Visualización comparativa de los resultados
df[['texto_original', 'texto_normalizado', 'texto_sin_stopwords', 'texto_stemmed']]

,texto_original,texto_normalizado,texto_sin_stopwords,texto_stemmed
0,¡Excelente explicación del algoritmo de NLP! M...,excelente explicación del algoritmo de nlp me ...,excelente explicación algoritmo nlp ayudó examen,excelent explic algoritm nlp ayud exam
1,El video es muy largo y la explicación de los ...,el video es muy largo y la explicación de los ...,video largo explicación transformers demasiado...,vide larg explic transformers demasi rap confus
2,¿Alguien sabe qué librería de Python es mejor ...,alguien sabe qué librería de python es mejor p...,alguien sabe librería python mejor sentiment a...,algui sab libr python mejor sentiment analysis...
3,"No me funcionó el código de Hugging Face, me s...",no me funcionó el código de hugging face me sa...,funcionó código hugging face sale error versió...,funcion codig hugging fac sal error version co...
4,Increíble contenido sobre Inteligencia Artific...,increíble contenido sobre inteligencia artific...,increíble contenido inteligencia artificial pl...,increibl conten inteligent artificial pln bien...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Lematización con el modelo de spaCy
def lematizar_texto(texto):
    doc = nlp(texto)
    lemmas = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(lemmas)

df['texto_lematizado'] = df['texto_normalizado'].apply(lematizar_texto)

# 2. Cálculo de TF-IDF (Term Frequency x Inverse Document Frequency)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['texto_lematizado'])

# Conversión a DataFrame para presentación limpia en el informe
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Comentario {i+1}" for i in range(len(df))]
)

# Impresión de resultados
print("=== COMPARACIÓN DE LEMATIZACIÓN ===")
print(df[['texto_original', 'texto_lematizado']])

print("\n=== MATRIZ TF-IDF (Muestra de Términos) ===")
print(df_tfidf.iloc[:, :8].round(3))  # Muestra las primeras 8 columnas

=== COMPARACIÓN DE LEMATIZACIÓN ===
                                      texto_original  \
0  ¡Excelente explicación del algoritmo de NLP! M...   
1  El video es muy largo y la explicación de los ...   
2  ¿Alguien sabe qué librería de Python es mejor ...   
3  No me funcionó el código de Hugging Face, me s...   
4  Increíble contenido sobre Inteligencia Artific...   

                                    texto_lematizado  
0  excelente explicación algoritmo nlp ayudar examen  
1       video explicación transformer rápido confuso  
2  alguien libreer python sentiment analysis 2026...  
3  funcionar código hugging facir salir error ver...  
4  increíble contenido inteligencia artificial pl...  

=== MATRIZ TF-IDF (Muestra de Términos) ===
               2026  algoritmo  alguien  analysis  artificial  ayudar  colab  \
Comentario 1  0.000      0.421    0.000     0.000       0.000   0.421  0.000   
Comentario 2  0.000      0.000    0.000     0.000       0.000   0.000  0.000   
Comentario 3

In [8]:
from spacy import displacy

# 1. Part-of-Speech (POS Tagging) y Named Entity Recognition (NER)
def analizar_sintaxis_y_entidades(texto):
    doc = nlp(texto)

    print(f"--- ANÁLISIS DE TEXTO: '{texto[:50]}...' ---")
    print("\n1. POS TAGGING (Primeros 6 tokens):")
    for token in list(doc)[:6]:
        print(f"Token: {token.text:<12} | POS: {token.pos_:<8} | Tag: {token.tag_}")

    print("\n2. NAMED ENTITY RECOGNITION (NER):")
    if doc.ents:
        for ent in doc.ents:
            print(f"Entidad: {ent.text:<20} | Etiqueta: {ent.label_}")
    else:
        print("No se detectaron entidades nombradas.")
    print("="*60)

# Ejecución sobre el dataset
for texto in df['texto_original']:
    analizar_sintaxis_y_entidades(texto)

--- ANÁLISIS DE TEXTO: '¡Excelente explicación del algoritmo de NLP! Me ay...' ---

1. POS TAGGING (Primeros 6 tokens):
Token: ¡            | POS: PUNCT    | Tag: PUNCT
Token: Excelente    | POS: ADJ      | Tag: ADJ
Token: explicación  | POS: NOUN     | Tag: NOUN
Token: del          | POS: ADP      | Tag: ADP
Token: algoritmo    | POS: NOUN     | Tag: NOUN
Token: de           | POS: ADP      | Tag: ADP

2. NAMED ENTITY RECOGNITION (NER):
Entidad: ¡Excelente explicación del algoritmo de | Etiqueta: MISC
Entidad: NLP! Me ayudó mucho para mi examen | Etiqueta: MISC
--- ANÁLISIS DE TEXTO: 'El video es muy largo y la explicación de los tran...' ---

1. POS TAGGING (Primeros 6 tokens):
Token: El           | POS: DET      | Tag: DET
Token: video        | POS: NOUN     | Tag: NOUN
Token: es           | POS: AUX      | Tag: AUX
Token: muy          | POS: ADV      | Tag: ADV
Token: largo        | POS: ADJ      | Tag: ADJ
Token: y            | POS: CCONJ    | Tag: CCONJ

2. NAMED ENTITY RECOGNITI

In [9]:
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob

# 1. Modelado de Temas con LDA
lda_model = LatentDirichletAllocation(n_components=2, random_state=42)
lda_model.fit(tfidf_matrix)

print("=== 1. MODELADO DE TEMAS (LDA) ===")
words = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [words[i] for i in topic.argsort()[:-5:-1]]
    print(f"Tema {topic_idx + 1}: {', '.join(top_words)}")

# 2. Análisis de Sentimientos con TextBlob
def obtener_sentimiento(texto):
    # Traducimos temporalmente para aprovechar el léxico calibrado de TextBlob
    blob = TextBlob(texto)
    try:
        blob_en = blob.translate(from_lang='es', to='en')
        polaridad = blob_en.sentiment.polarity
    except:
        polaridad = blob.sentiment.polarity

    if polaridad > 0.1:
        return "Positivo", round(polaridad, 2)
    elif polaridad < -0.1:
        return "Negativo", round(polaridad, 2)
    else:
        return "Neutro", round(polaridad, 2)

df[['sentimiento', 'polaridad']] = df['texto_original'].apply(lambda x: pd.Series(obtener_sentimiento(x)))

print("\n=== 2. CLASIFICACIÓN Y ANÁLISIS DE SENTIMIENTOS ===")
print(df[['usuario', 'texto_original', 'sentimiento', 'polaridad']])

=== 1. MODELADO DE TEMAS (LDA) ===
Tema 1: 2026, colombia, saludo, python
Tema 2: explicación, transformer, rápido, confuso

=== 2. CLASIFICACIÓN Y ANÁLISIS DE SENTIMIENTOS ===
       usuario                                     texto_original sentimiento  \
0  tech_user99  ¡Excelente explicación del algoritmo de NLP! M...      Neutro   
1   dev_carlos  El video es muy largo y la explicación de los ...      Neutro   
2       ana_ai  ¿Alguien sabe qué librería de Python es mejor ...      Neutro   
3  code_master  No me funcionó el código de Hugging Face, me s...      Neutro   
4      laura_m  Increíble contenido sobre Inteligencia Artific...    Negativo   

   polaridad  
0        0.0  
1        0.0  
2        0.0  
3        0.0  
4       -0.6  


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Búsqueda y Recuperación de Información con Similitud de Coseno
consulta = "tutorial de inteligencia artificial y librerias de python"
consulta_lematizada = lematizar_texto(normalizar_texto(consulta))

# Vectorizar la consulta con el mismo TfidfVectorizer
consulta_vector = vectorizer.transform([consulta_lematizada])

# Calcular similitud del coseno entre la consulta y todos los comentarios
similitudes = cosine_similarity(consulta_vector, tfidf_matrix).flatten()

# Agregar las puntuaciones de similitud al DataFrame
df['similitud_consulta'] = similitudes.round(3)

print("=== 1. RECUPERACIÓN DE INFORMACIÓN (Similitud con la consulta) ===")
print(f"Consulta ingresada: '{consulta}'")
print(df[['usuario', 'texto_original', 'similitud_consulta']].sort_values(by='similitud_consulta', ascending=False))

# 2. Obtención de Word Embeddings (Vectores densos con spaCy)
print("\n=== 2. WORD EMBEDDINGS (Vectores continuos de 96 dimensiones) ===")
# Obtenemos el embedding promedio del primer comentario
doc_emb = nlp(df['texto_normalizado'].iloc[0])
vector_denso = doc_emb.vector

print(f"Comentario evaluado: '{df['texto_original'].iloc[0][:40]}...'")
print(f"Dimensión del vector de embedding: {vector_denso.shape}")
print(f"Muestra de las primeras 8 dimensiones del vector:\n{np.round(vector_denso[:8], 4)}")

=== 1. RECUPERACIÓN DE INFORMACIÓN (Similitud con la consulta) ===
Consulta ingresada: 'tutorial de inteligencia artificial y librerias de python'
       usuario                                     texto_original  \
4      laura_m  Increíble contenido sobre Inteligencia Artific...   
2       ana_ai  ¿Alguien sabe qué librería de Python es mejor ...   
3  code_master  No me funcionó el código de Hugging Face, me s...   
0  tech_user99  ¡Excelente explicación del algoritmo de NLP! M...   
1   dev_carlos  El video es muy largo y la explicación de los ...   

   similitud_consulta  
4               0.354  
2               0.177  
3               0.158  
0               0.000  
1               0.000  

=== 2. WORD EMBEDDINGS (Vectores continuos de 96 dimensiones) ===
Comentario evaluado: '¡Excelente explicación del algoritmo de ...'
Dimensión del vector de embedding: (96,)
Muestra de las primeras 8 dimensiones del vector:
[ 0.053  -0.7892 -0.9605 -0.4986  0.4729 -1.4496  1.4035  0.3527]


In [11]:
# 1. Resumen Automático Extractivo
def generar_resumen(df_input):
    textos_clave = df_input[df_input['similitud_consulta'] > 0]['texto_original'].tolist()
    resumen = " | ".join(textos_clave)
    return resumen

resumen_ejecutivo = generar_resumen(df)

print("=== 1. RESUMEN AUTOMÁTICO DE LOS COMENTARIOS MÁS RELEVANTES ===")
print(f"Resumen Sintetizado:\n\"{resumen_ejecutivo}\"")

# 2. Reconocimiento de Patrones con Expresiones Regulares
print("\n=== 2. RECONOCIMIENTO DE PATRONES (Menciones a Tecnologías / Años) ===")
patron_tec = r'\b(Python|Hugging Face|Colab|NLP|Transformers)\b'
patron_año = r'\b(20\d{2})\b'

for idx, texto in enumerate(df['texto_original']):
    tec_encontradas = re.findall(patron_tec, texto, re.IGNORECASE)
    años_encontrados = re.findall(patron_año, texto)
    if tec_encontradas or años_encontrados:
        print(f"Comentario {idx+1}: Tecnologías -> {tec_encontradas} | Años -> {años_encontrados}")

# 3. Vista final del DataFrame consolidado
print("\n=== 3. DATAFRAME FINAL DE TRABAJO (17 Procesos de PLN) ===")
df[['id_comentario', 'usuario', 'texto_original', 'texto_lematizado', 'sentimiento', 'similitud_consulta']]

=== 1. RESUMEN AUTOMÁTICO DE LOS COMENTARIOS MÁS RELEVANTES ===
Resumen Sintetizado:
"¿Alguien sabe qué librería de Python es mejor para sentiment analysis en 2026? Saludos desde Colombia. | No me funcionó el código de Hugging Face, me sale un error de versión en Colab. Pésimo tutorial. | Increíble contenido sobre Inteligencia Artificial y PLN. Muy bien estructurado y fácil de seguir."

=== 2. RECONOCIMIENTO DE PATRONES (Menciones a Tecnologías / Años) ===
Comentario 1: Tecnologías -> ['NLP'] | Años -> []
Comentario 2: Tecnologías -> ['transformers'] | Años -> []
Comentario 3: Tecnologías -> ['Python'] | Años -> ['2026']
Comentario 4: Tecnologías -> ['Hugging Face', 'Colab'] | Años -> []

=== 3. DATAFRAME FINAL DE TRABAJO (17 Procesos de PLN) ===


,id_comentario,usuario,texto_original,texto_lematizado,sentimiento,similitud_consulta
0,101,tech_user99,¡Excelente explicación del algoritmo de NLP! M...,excelente explicación algoritmo nlp ayudar examen,Neutro,0.000
1,102,dev_carlos,El video es muy largo y la explicación de los ...,video explicación transformer rápido confuso,Neutro,0.000
2,103,ana_ai,¿Alguien sabe qué librería de Python es mejor ...,alguien libreer python sentiment analysis 2026...,Neutro,0.177
3,104,code_master,"No me funcionó el código de Hugging Face, me s...",funcionar código hugging facir salir error ver...,Neutro,0.158
4,105,laura_m,Increíble contenido sobre Inteligencia Artific...,increíble contenido inteligencia artificial pl...,Negativo,0.354


In [12]:
import pandas as pd
import numpy as np

# 1. Matriz TF (Term Frequency) relativa por documento
tf_matrix = np.divide(tfidf_matrix.toarray(), tfidf_matrix.toarray().sum(axis=1, keepdims=True),
                      out=np.zeros_like(tfidf_matrix.toarray()), where=tfidf_matrix.toarray().sum(axis=1, keepdims=True)!=0)

df_tf = pd.DataFrame(tf_matrix, columns=vectorizer.get_feature_names_out())

# 2. Valores IDF (Inverse Document Frequency) asignados a cada término
df_idf = pd.DataFrame({
    'término': vectorizer.get_feature_names_out(),
    'idf': vectorizer.idf_
}).sort_values(by='idf', ascending=False)

print("=== 1. MATRIZ DE FRECUENCIA DE TÉRMINO (TF) ===")
print(df_tf.iloc[:, :6].round(3))

print("\n=== 2. VALORES DE FRECUENCIA INVERSA DE DOCUMENTO (IDF) ===")
print(df_idf.head(8))

=== 1. MATRIZ DE FRECUENCIA DE TÉRMINO (TF) ===
    2026  algoritmo  alguien  analysis  artificial  ayudar
0  0.000      0.172    0.000     0.000       0.000   0.172
1  0.000      0.000    0.000     0.000       0.000   0.000
2  0.125      0.000    0.125     0.125       0.000   0.000
3  0.000      0.000    0.000     0.000       0.000   0.000
4  0.000      0.000    0.000     0.000       0.125   0.000

=== 2. VALORES DE FRECUENCIA INVERSA DE DOCUMENTO (IDF) ===
      término       idf
0        2026  2.098612
1   algoritmo  2.098612
2     alguien  2.098612
3    analysis  2.098612
4  artificial  2.098612
5      ayudar  2.098612
6       colab  2.098612
7    colombia  2.098612


In [13]:
doc_parsing = nlp(df['texto_original'].iloc[0])

print("=== PARSING SINTÁCTICO Y DEPENDENCIAS GRAMATICALES ===")
print(f"Texto evaluado: '{df['texto_original'].iloc[0]}'\n")
print(f"{'Texto':<15} | {'Función (Dep)':<15} | {'Palabra Principal (Head)':<20} | {'POS Head':<10}")
print("-" * 68)

for token in doc_parsing:
    print(f"{token.text:<15} | {token.dep_:<15} | {token.head.text:<20} | {token.head.pos_:<10}")

=== PARSING SINTÁCTICO Y DEPENDENCIAS GRAMATICALES ===
Texto evaluado: '¡Excelente explicación del algoritmo de NLP! Me ayudó mucho para mi examen. http://youtube.com/watch?v=123'

Texto           | Función (Dep)   | Palabra Principal (Head) | POS Head  
--------------------------------------------------------------------
¡               | punct           | explicación          | NOUN      
Excelente       | amod            | explicación          | NOUN      
explicación     | nsubj           | ayudó                | VERB      
del             | case            | algoritmo            | NOUN      
algoritmo       | nmod            | explicación          | NOUN      
de              | case            | NLP                  | PROPN     
NLP             | nmod            | algoritmo            | NOUN      
!               | punct           | explicación          | NOUN      
Me              | iobj            | ayudó                | VERB      
ayudó           | ROOT            | ayudó     

In [14]:
from transformers import pipeline

# Cargar pipeline de Text Generation con un modelo liviano
generator = pipeline('text-generation', model='distilgpt2')

prompt = "Artificial Intelligence and Natural Language Processing are"
resultado_llm = generator(prompt, max_length=45, num_return_sequences=1)

print("=== GENERACIÓN DE TEXTO CON MODELO LLM (Transformers / DistilGPT2) ===")
print(f"Prompt Inicial: '{prompt}'")
print(f"Texto Generado:\n{resultado_llm[0]['generated_text']}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=45) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== GENERACIÓN DE TEXTO CON MODELO LLM (Transformers / DistilGPT2) ===
Prompt Inicial: 'Artificial Intelligence and Natural Language Processing are'
Texto Generado:
Artificial Intelligence and Natural Language Processing are also at the forefront of the AI debate. In this article, we explore the potential of artificial intelligence to help solve the problem of human interaction. In this article, we discuss how artificial intelligence can help people solve problems in everyday life.


In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Cargar modelo y tokenizador de QA explícitamente
model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

contexto = df['texto_original'].iloc[3] # "No me funcionó el código de Hugging Face..."
pregunta = "What platform or library is having an error?"

# Tokenizar entrada
inputs = tokenizer(pregunta, contexto, return_tensors="pt")

# Obtener logits de respuesta
with torch.no_grad():
    outputs = model(**inputs)

# Extraer el fragmento de texto con las probabilidades más altas
start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1
answer_tokens = inputs["input_ids"][0][start_idx:end_idx]
respuesta_texto = tokenizer.decode(answer_tokens)

print("=== QUESTION ANSWERING (Extracción de Respuestas sobre el Corpus) ===")
print(f"Contexto: '{contexto}'")
print(f"Pregunta: '{pregunta}'")
print(f"Respuesta Encontrada: '{respuesta_texto}'")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

=== QUESTION ANSWERING (Extracción de Respuestas sobre el Corpus) ===
Contexto: 'No me funcionó el código de Hugging Face, me sale un error de versión en Colab. Pésimo tutorial.'
Pregunta: 'What platform or library is having an error?'
Respuesta Encontrada: ''


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Cargar modelo y tokenizador de Summarization
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer_sum = AutoTokenizer.from_pretrained(model_name)
model_sum = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Unificar los comentarios para generar un resumen abstracto global
texto_largo = " ".join(df['texto_original'].tolist())

# 2. Tokenización del texto de entrada
inputs = tokenizer_sum("summarize: " + texto_largo, return_tensors="pt", max_length=1024, truncation=True)

# 3. Generación del resumen abstracto
with torch.no_grad():
    summary_ids = model_sum.generate(
        inputs["input_ids"],
        max_length=50,
        min_length=15,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

resumen_real = tokenizer_sum.decode(summary_ids[0], skip_special_tokens=True)

print("=== 5. SUMMARIZATION REAL (Resumen Neuronal Abstractivo con DistilBART) ===")
print(f"Texto de Entrada Unificado:\n\"{texto_largo}\"\n")
print(f"Resumen Neuronal Generado:\n\"{resumen_real}\"")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

=== 5. SUMMARIZATION REAL (Resumen Neuronal Abstractivo con DistilBART) ===
Texto de Entrada Unificado:
"¡Excelente explicación del algoritmo de NLP! Me ayudó mucho para mi examen. http://youtube.com/watch?v=123 El video es muy largo y la explicación de los transformers fue demasiado rápida y confusa. ¿Alguien sabe qué librería de Python es mejor para sentiment analysis en 2026? Saludos desde Colombia. No me funcionó el código de Hugging Face, me sale un error de versión en Colab. Pésimo tutorial. Increíble contenido sobre Inteligencia Artificial y PLN. Muy bien estructurado y fácil de seguir."

Resumen Neuronal Generado:
" The video is muy largo y la explicaciación de los transformers fue demasiado rápida y confusa . No me funcionó el código de Hugging Face, me sale"


In [19]:
from sentence_transformers import SentenceTransformer, util

# 1. Cargar el modelo especializado en embeddings de oraciones
model_st = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Seleccionar dos oraciones distintas del corpus para comparar
oracion_1 = df['texto_original'].iloc[0] # Comentario de tech_user99
oracion_2 = df['texto_original'].iloc[4] # Comentario de laura_m

# 3. Generar los embeddings vectoriales
emb1 = model_st.encode(oracion_1, convert_to_tensor=True)
emb2 = model_st.encode(oracion_2, convert_to_tensor=True)

# 4. Calcular la similitud del coseno par a par
similitud_oraciones = util.cos_sim(emb1, emb2).item()

print("=== 6. SENTENCE SIMILARITY (Similitud Semántica entre Oraciones) ===")
print(f"Oración 1: \"{oracion_1}\"")
print(f"Oración 2: \"{oracion_2}\"")
print(f"Puntuación de Similitud Semántica Directa: {round(similitud_oraciones, 4)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== 6. SENTENCE SIMILARITY (Similitud Semántica entre Oraciones) ===
Oración 1: "¡Excelente explicación del algoritmo de NLP! Me ayudó mucho para mi examen. http://youtube.com/watch?v=123"
Oración 2: "Increíble contenido sobre Inteligencia Artificial y PLN. Muy bien estructurado y fácil de seguir."
Puntuación de Similitud Semántica Directa: 0.3727


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Cargar el modelo y tokenizador de traducción (Español a Inglés)
model_trans_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer_trans = AutoTokenizer.from_pretrained(model_trans_name)
model_trans = AutoModelForSeq2SeqLM.from_pretrained(model_trans_name)

# 2. Seleccionar un comentario en español del corpus
texto_es = df['texto_original'].iloc[0]

# 3. Tokenizar y generar la traducción
inputs_trans = tokenizer_trans(texto_es, return_tensors="pt", padding=True)

with torch.no_grad():
    translated_ids = model_trans.generate(**inputs_trans)

texto_en = tokenizer_trans.decode(translated_ids[0], skip_special_tokens=True)

print("=== 7. TRADUCCIÓN AUTOMÁTICA DE TEXTO (Translation ES -> EN) ===")
print(f"Texto Original (Español): \"{texto_es}\"")
print(f"Texto Traducido (Inglés) : \"{texto_en}\"")

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

=== 7. TRADUCCIÓN AUTOMÁTICA DE TEXTO (Translation ES -> EN) ===
Texto Original (Español): "¡Excelente explicación del algoritmo de NLP! Me ayudó mucho para mi examen. http://youtube.com/watch?v=123"
Texto Traducido (Inglés) : "Excellent explanation of the NLP algorithm! It helped me a lot for my exam. http://youtube.com/watch?v=123"


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Cargar el modelo y tokenizador de Generación de Texto (LLM Causal)
model_gen_name = "distilgpt2"
tokenizer_gen = AutoTokenizer.from_pretrained(model_gen_name)
model_gen = AutoModelForCausalLM.from_pretrained(model_gen_name)

# 2. Definir un prompt en inglés
prompt = "Artificial Intelligence and Natural Language Processing are"

# 3. Tokenizar y generar la continuación del texto
inputs_gen = tokenizer_gen(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = model_gen.generate(
        **inputs_gen,
        max_length=45,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer_gen.eos_token_id
    )

texto_generado = tokenizer_gen.decode(output_ids[0], skip_special_tokens=True)

print("=== 8. GENERACIÓN DE TEXTO CON MODELO LLM (DistilGPT2) ===")
print(f"Prompt Inicial: \"{prompt}\"")
print(f"Texto Generado:\n\"{texto_generado}\"")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

=== 8. GENERACIÓN DE TEXTO CON MODELO LLM (DistilGPT2) ===
Prompt Inicial: "Artificial Intelligence and Natural Language Processing are"
Texto Generado:
"Artificial Intelligence and Natural Language Processing are two of the most widely used, and most popular, artificial intelligence systems. The first is artificial Intelligence, which is a computer program which learns, processes, uses, or manipulates human"


In [22]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 1. Extracción de Bigramas (pares de palabras consecutivas más frecuentes)
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_counts = bigram_vectorizer.fit_transform(df['texto_lematizado'])

# 2. Consolidación en un DataFrame analítico de Text Mining
df_bigramas = pd.DataFrame({
    'Patrón / Bigrama': bigram_vectorizer.get_feature_names_out(),
    'Frecuencia Absoluta': bigram_counts.toarray().sum(axis=0)
}).sort_values(by='Frecuencia Absoluta', ascending=False).reset_index(drop=True)

print("=== 9. DEMOSTRACIÓN DE TEXT MINING (Co-ocurrencia de N-Gramas) ===")
print(df_bigramas.head(8))

=== 9. DEMOSTRACIÓN DE TEXT MINING (Co-ocurrencia de N-Gramas) ===
         Patrón / Bigrama  Frecuencia Absoluta
0             2026 saludo                    1
1           algoritmo nlp                    1
2         alguien libreer                    1
3           analysis 2026                    1
4          artificial pln                    1
5           ayudar examen                    1
6            colab pésimo                    1
7  contenido inteligencia                    1
